# 06 — Movie Revenue Prediction

Goal: predict movie gross revenue.

The target is highly right-skewed, so the model predicts:

`log(1 + gross)`

and predictions are transformed back to the original revenue scale for evaluation.

Important leakage rule:
- Do not use gross, profit, ROI, or any feature calculated from gross as an input.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

movies = pd.read_csv("../data/processed/movies_cleaned.csv")
movies = movies.dropna(how="all").reset_index(drop=True)

for col in ["year", "score", "votes", "budget", "gross", "runtime", "release_year"]:
    movies[col] = pd.to_numeric(movies[col], errors="coerce")

for col in ["genre", "rating", "country", "release_month"]:
    movies[col] = movies[col].fillna("Unknown")

features = [
    "year", "score", "votes", "budget", "runtime",
    "release_year", "genre", "rating", "country", "release_month"
]

data = movies[features + ["gross"]].dropna(subset=["gross"]).copy()

X = data[features]
y = np.log1p(data["gross"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_features = [
    "year", "score", "votes", "budget", "runtime", "release_year"
]
categorical_features = ["genre", "rating", "country", "release_month"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(
    preprocessor.fit_transform(X_train),
    y_train
)

baseline_log_pred = baseline.predict(preprocessor.transform(X_test))
baseline_pred = np.expm1(baseline_log_pred)
actual = np.expm1(y_test)

print("Baseline MAE:", round(mean_absolute_error(actual, baseline_pred), 2))
print("Baseline RMSE:", round(np.sqrt(mean_squared_error(actual, baseline_pred)), 2))
print("Baseline R²:", round(r2_score(actual, baseline_pred), 4))

In [ ]:
revenue_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2,
        max_features="sqrt"
    ))
])

revenue_model.fit(X_train, y_train)

pred_log = revenue_model.predict(X_test)
pred_revenue = np.expm1(pred_log)
actual_revenue = np.expm1(y_test)

revenue_metrics = {
    "MAE": mean_absolute_error(actual_revenue, pred_revenue),
    "RMSE": np.sqrt(mean_squared_error(actual_revenue, pred_revenue)),
    "R2": r2_score(actual_revenue, pred_revenue)
}

pd.Series(revenue_metrics)

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Baseline", "Random Forest"],
    "MAE": [
        mean_absolute_error(actual_revenue, baseline_pred),
        mean_absolute_error(actual_revenue, pred_revenue)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(actual_revenue, baseline_pred)),
        np.sqrt(mean_squared_error(actual_revenue, pred_revenue))
    ],
    "R2": [
        r2_score(actual_revenue, baseline_pred),
        r2_score(actual_revenue, pred_revenue)
    ]
})

comparison

In [ ]:
revenue_results = pd.DataFrame({
    "actual_gross": actual_revenue.values,
    "predicted_gross": pred_revenue
})

revenue_results.head(20)

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    x=np.log1p(revenue_results["actual_gross"]),
    y=np.log1p(revenue_results["predicted_gross"]),
    alpha=0.5
)

min_value = min(
    np.log1p(revenue_results["actual_gross"]).min(),
    np.log1p(revenue_results["predicted_gross"]).min()
)
max_value = max(
    np.log1p(revenue_results["actual_gross"]).max(),
    np.log1p(revenue_results["predicted_gross"]).max()
)

plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.title("Actual vs Predicted Gross Revenue")
plt.xlabel("Actual log(1 + Gross)")
plt.ylabel("Predicted log(1 + Gross)")
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("../models", exist_ok=True)
joblib.dump(revenue_model, "../models/revenue_model.pkl")
print("Saved: ../models/revenue_model.pkl")

## Important interpretation

Revenue is influenced by many factors not contained in this dataset. The model should be presented as an experiment in predictive modeling, not as a production financial forecasting system.